# Week 17: Retrieval-Augmented Generation (RAG) - Part 1

## Agentic RAG with Strands + Amazon Bedrock Knowledge Bases

## Learning Objectives

By the end of this session, you will be able to:
1. **Explain agentic RAG** - when retrieval becomes a tool the agent decides to call, not a one-shot pipeline step
2. **Wire a Bedrock Knowledge Base into a Strands agent** using `strands_tools.retrieve` and consume citations from the retrieval results
3. **Use FAISS-backed agent memory** via `strands_tools.mem0_memory` for persistent, cross-session knowledge your agents can store and recall
4. **Build a multi-retriever supervisor** that routes questions to specialist retriever agents - extending the Week 16 fraud pipeline to replace hard-coded policies with a queryable knowledge base

## Prerequisites

- Completed Weeks 11-16 (LLM APIs, evaluation, Bedrock, fine-tuning, agents with LangChain, Strands agents + supervisor pattern)
- Comfortable with the Week 16 `@tool` decorator, `Agent(model=..., tools=[...], system_prompt=...)` constructor, and the agents-as-tools pattern
- AWS credentials via SageMaker execution role (same as Week 13 / 15 / 16)
- Watched pre-class videos on RAG concepts, embedding models, vector databases

## Session Format (~2 hours)

| Section | Duration | Type |
|---------|----------|------|
| Section 0: Setup & Week 15/16 Recap | 10 min | Code |
| Section 1: Agentic RAG vs Naive RAG | 15 min | Demo |
| Section 2: Bedrock KB via `strands_tools.retrieve` | 20 min | Demo + Lab 1 |
| Section 3: FAISS-Backed Agent Memory | 20 min | Demo + Lab 2 |
| Section 4: Multi-Retriever Supervisor (MAIN OUTCOME) | 30 min | Demo + Main Lab 3 |
| Wrap-up & Homework | 5 min | Markdown |

## The Story So Far

In Week 13 you called a Bedrock Knowledge Base directly with `retrieveAndGenerate` - a one-shot Q&A service. In Weeks 15-16 you built agents that reason, call tools, and delegate to specialist sub-agents. But all the specialist knowledge was hard-coded - remember the `FRAUD_POLICIES` dict with 7 hand-written rules?

Today that dictionary grows up. Fraud policy at a real financial institution isn't 7 rules - it's hundreds of pages of BSA/AML regulations, OFAC sanctions guidance, FFIEC examination manuals, and internal procedures that change monthly. You can't paste that into a Python dict. You have to put it in a knowledge base and let your agents **decide when and how to query it**.

That's agentic RAG. And because your fraud pipeline already has multiple specialist agents, you don't give retrieval to ONE agent - you create **specialist retriever agents** and let the supervisor route:

```mermaid
graph TD
    USER[User: investigate TXN-001]
    SUP[Supervisor Agent]

    TRIAGE[Triage Agent<br/>Week 16]
    INV[Investigation Agent<br/>Week 16]
    POLICY[PolicyRetriever<br/>Week 17 NEW<br/>Bedrock KB]
    CASE[CaseHistoryRetriever<br/>Week 17 NEW<br/>FAISS mem0]
    DEC[Decision Agent<br/>Week 16]

    KB[(Bedrock Knowledge Base<br/>Fraud policies, BSA/AML,<br/>OFAC, FFIEC docs)]
    MEM[(FAISS-backed mem0<br/>Past case outcomes)]

    USER --> SUP
    SUP --> TRIAGE
    SUP --> INV
    SUP --> POLICY
    SUP --> CASE
    SUP --> DEC

    POLICY --> KB
    CASE --> MEM

    style POLICY fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
    style CASE fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
    style KB fill:#fff3e0,stroke:#ff9800
    style MEM fill:#fff3e0,stroke:#ff9800
```

The green nodes are what we build today. The supervisor learns to route.

## This Week vs Next Week

| | **Week 17 (Today)** | **Week 18 (Next Week)** |
|---|---|---|
| **Focus** | Retrieval as an agent tool | What's INSIDE the retrieval pipeline |
| **You build** | Multi-retriever supervisor with Bedrock KB + FAISS mem0 | Evaluate and tune: chunking, reranking, RAGAS |
| **Key concepts** | `strands_tools.retrieve`, `mem0_memory`, multi-retriever routing | Chunk sizes, cross-encoders, faithfulness/answer-relevancy metrics |
| **Why it matters** | Your agents can look things up now | You can measure whether they're looking up the RIGHT things |

## GPU Setup

No GPU needed - all work is API-based through Amazon Bedrock. A CPU SageMaker Studio notebook is sufficient.

# Section 0: Environment Setup & Week 15/16 Recap

We'll use the **Strands Agents SDK** (from Week 16), plus two new `strands_tools`:
- `retrieve` - wraps the Bedrock Knowledge Base `Retrieve` API so an agent can query the KB as a tool
- `mem0_memory` - FAISS-backed persistent agent memory for per-agent case notes

AWS credentials come from your SageMaker execution role - same pattern as Week 13 / 15 / 16. No `getpass`, no API keys to paste.

Before running the notebook, the **instructor** must have:
1. Pre-created a Bedrock Knowledge Base (using `instructor_setup_kb.py` in this folder) and distributed the KB ID via the `STRANDS_KNOWLEDGE_BASE_ID` environment variable.
2. Enabled Bedrock model access for `anthropic.claude-sonnet-4-5-20250929-v1:0` and `amazon.titan-embed-text-v2:0`.

The setup cells below will verify BOTH of those and fail loud if anything is missing.

In [ ]:
# =============================================================================
# INSTALL REQUIRED LIBRARIES
# =============================================================================
# strands-agents                       : Core SDK (from Week 16)
# strands-agents-tools[mem0-memory]    : retrieve + mem0_memory (brings in
#                                         mem0ai, rank_bm25, opensearch-py)
#                                         >= 0.2.10 required: earlier versions
#                                         do not support MEM0_LLM_MODEL env var
# boto3                                 : AWS SDK (bedrock-agent-runtime)
# faiss-cpu                             : local FAISS index used by mem0_memory
# numpy<2                               : FAISS is NOT compatible with numpy 2.x
#                                         (known upstream issue)
# sagemaker is pre-installed on SageMaker Studio - do not reinstall.

%pip install -q \
    "strands-agents>=1.37,<2" \
    "strands-agents-tools[mem0-memory]>=0.2.10" \
    "boto3>=1.35" \
    "faiss-cpu>=1.8,<2" \
    "rank_bm25>=0.2.2" \
    "opensearch-py>=2.4" \
    "numpy<2"

print("\nPackages installed.")
print("If this was your first install, RESTART THE KERNEL before running the next cell.")

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
# Import order matters: we import strands_tools BEFORE any direct faiss call
# to avoid segfaults some kernels hit when faiss is imported before
# sentence-transformers. Strands tools pulls sentence-transformers for us.
#
# IMPORTANT: strands_tools.retrieve reads env vars AT IMPORT TIME. We set the
# required ones (KNOWLEDGE_BASE_ID, MIN_SCORE, RETRIEVE_ENABLE_METADATA_DEFAULT)
# BEFORE the import below, so the first call to retrieve works without
# needing a kernel restart.

import os
import json
import boto3
import sagemaker
from sagemaker import get_execution_role
from importlib.metadata import version as pkg_version


# =============================================================================
# SET strands_tools.retrieve ENVIRONMENT VARIABLES (before import)
# =============================================================================
# These are the EXACT env var names strands_tools.retrieve reads (verified
# against the source). Setting them here makes the tool usable without any
# kernel restart or startup-file trick.

# KB id (Section 2/3/4 use this). Set to the class KB provisioned by the
# instructor. Students: keep this value.
os.environ["KNOWLEDGE_BASE_ID"] = os.environ.get("KNOWLEDGE_BASE_ID") \
    or os.environ.get("STRANDS_KNOWLEDGE_BASE_ID") \
    or "FARSQGTONR"  # class KB id (instructor-provisioned)

# Also set STRANDS_KNOWLEDGE_BASE_ID so downstream code that references it
# (the naive_rag cell, for example) keeps working.
os.environ["STRANDS_KNOWLEDGE_BASE_ID"] = os.environ["KNOWLEDGE_BASE_ID"]

# Lower the default minimum score so broader policy questions match.
# Default in strands_tools is 0.4; compliance corpus benefits from 0.2.
os.environ["MIN_SCORE"] = os.environ.get("MIN_SCORE", "0.2")

# Include source URIs + metadata in every retrieve result by default.
os.environ["RETRIEVE_ENABLE_METADATA_DEFAULT"] = "true"


# =============================================================================
# FIX FOR mem0_memory: SET MEM0_LLM_MODEL BEFORE THE IMPORT BELOW
# =============================================================================
# mem0_memory internally uses an LLM for memory summarization. The DEFAULT_CONFIG
# class-level dict in Mem0ServiceClient is evaluated ONCE at module import time,
# not at instantiation time. The default model ID (on-demand) is not accessible
# in this account - only inference profiles (us.* prefix) are.
#
# SOLUTION: set MEM0_LLM_MODEL in the environment BEFORE the import line below.
# strands_tools.__init__.py is empty, so mem0_memory is only loaded when we
# explicitly import it - this env var will be present at the right moment.
#
# Do NOT move this line below the import. The ordering is the fix.
os.environ["MEM0_LLM_MODEL"] = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"


# =============================================================================
# STRANDS IMPORTS (after env vars are set)
# =============================================================================
from strands import Agent, tool
from strands.models import BedrockModel

# NEW in Week 17: retrieve (Bedrock KB) and mem0_memory (FAISS-backed)
from strands_tools import retrieve, mem0_memory


# =============================================================================
# BELT-AND-SUSPENDERS: PATCH mem0_memory DEFAULT_CONFIG AFTER IMPORT
# =============================================================================
# In case the module was already in sys.modules before the env var was set
# (e.g. kernel imported it transitively), we also patch the baked-in dict
# directly. This is safe to run unconditionally - it is idempotent.
from strands_tools.mem0_memory import Mem0ServiceClient
Mem0ServiceClient.DEFAULT_CONFIG["llm"]["config"]["model"] = os.environ["MEM0_LLM_MODEL"]


# Version check (use importlib.metadata, NOT pkg.__version__)
for pkg in ["strands-agents", "strands-agents-tools", "boto3", "faiss-cpu", "numpy"]:
    try:
        print(f"  {pkg:28s} {pkg_version(pkg)}")
    except Exception:
        print(f"  {pkg:28s} (not installed)")


# =============================================================================
# SAGEMAKER SESSION + EXECUTION ROLE (same pattern as Week 15/16)
# =============================================================================
# On SageMaker, AWS credentials come automatically through the execution role.
# No getpass, no manual credentials. The IAM role attached to this notebook
# instance already has permissions for Bedrock and S3.

sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name

# Make region available to strands_tools.retrieve and mem0_memory, which read
# it from the environment.
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

print(f"\nSageMaker execution role: {role.split('/')[-1]}")
print(f"AWS Region:               {AWS_REGION}")


# =============================================================================
# VERIFY BEDROCK CONNECTION (fail loud if anything is off)
# =============================================================================
bedrock_client = boto3.client("bedrock", region_name=AWS_REGION)
try:
    models = bedrock_client.list_foundation_models()
    print(f"\nConnected to Bedrock: {len(models['modelSummaries'])} models visible.")
except Exception as e:
    print(f"\nBedrock connection failed: {e}")
    print("Ask your instructor to verify the execution role has Bedrock permissions.")
    raise


# =============================================================================
# MODEL CONFIGURATION (same as Week 15/16 - continuity)
# =============================================================================
# Claude 3 Haiku via inference profile - fast, cheap, supports tool use.
# This is the EXACT model ID the Week 15-16 notebooks use.
MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

# Titan Text Embeddings V2 is the embedding model used by the shared class KB.
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"

# Build the Strands BedrockModel. Strands will use the SageMaker role's
# credentials automatically (boto3 default credential chain).
llm = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION)

print(f"\nAgent LLM:       {MODEL_ID}")
print(f"Embedding model: {EMBED_MODEL_ID}")
print(f"mem0 LLM model:  {Mem0ServiceClient.DEFAULT_CONFIG['llm']['config']['model']}")


# =============================================================================
# PROBE THE LLM - a tiny converse() call confirms model access BEFORE agent work
# =============================================================================
# Students have been burned by silent permission gaps. Better to fail here
# with a clear error than halfway through an agent loop.
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
try:
    probe = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 10, "temperature": 0},
    )
    probe_text = probe["output"]["message"]["content"][0]["text"]
    print(f"LLM probe OK:    {probe_text!r}")
except Exception as e:
    print(f"LLM probe FAILED: {e}")
    print(f"Ask your instructor to enable Bedrock access for {MODEL_ID}.")
    raise


# =============================================================================
# WEEK 17 SHARED KNOWLEDGE BASE ID
# =============================================================================
STRANDS_KNOWLEDGE_BASE_ID = os.environ["KNOWLEDGE_BASE_ID"]

# Verify the KB resolves - fail loud if the id is wrong.
bedrock_agent = boto3.client("bedrock-agent", region_name=AWS_REGION)
try:
    kb_info = bedrock_agent.get_knowledge_base(
        knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID
    )
    kb_name = kb_info["knowledgeBase"]["name"]
    print(f"\nKnowledge Base:  {STRANDS_KNOWLEDGE_BASE_ID}  ({kb_name})")
except Exception as e:
    print(f"\nKnowledge Base lookup FAILED: {e}")
    print("Double-check the KB id with your instructor.")
    raise

print("\nEnvironment ready.")

In [ ]:
# =============================================================================
# BACKUP: case_memory - Custom FAISS + Titan tool (use if mem0_memory fails)
# =============================================================================
# INSTRUCTOR: If mem0_memory throws a ValidationException about a model ID,
# uncomment the entire block below and replace tools=[mem0_memory] with
# tools=[case_memory] in the demo and lab cells in Section 3.
# The interface is identical: store | retrieve | list, user_id scoping.
# No hidden LLM dependency - embeddings only via Titan V2.
#
# import uuid, numpy as np, faiss, json, boto3
# from strands import tool
#
# _indexes: dict = {}
# _metadata: dict = {}
#
# def _embed(text, region):
#     client = boto3.client("bedrock-runtime", region_name=region)
#     resp = client.invoke_model(
#         modelId="amazon.titan-embed-text-v2:0",
#         body=json.dumps({"inputText": text}),
#         contentType="application/json", accept="application/json",
#     )
#     vec = np.array(json.loads(resp["body"].read())["embedding"], dtype="float32")
#     vec /= np.linalg.norm(vec) + 1e-9
#     return vec
#
# @tool
# def case_memory(action: str, user_id: str = "fraud-team-v1",
#                 content: str = "", query: str = "", top_k: int = 5) -> str:
#     """Store and retrieve fraud case outcomes via FAISS + Titan embeddings.
#     Actions: store | retrieve | list.
#     Args:
#         action: store, retrieve, or list.
#         user_id: Scope for isolation (default: fraud-team-v1).
#         content: Case note text (required for store).
#         query: Search query (required for retrieve).
#         top_k: Max results (default: 5).
#     """
#     region = os.environ.get("AWS_REGION", "us-east-1")
#     if action == "store":
#         if not content:
#             return "ERROR: content is required for store."
#         vec = _embed(content, region)
#         if user_id not in _indexes:
#             _indexes[user_id] = faiss.IndexFlatIP(1024)
#             _metadata[user_id] = []
#         _indexes[user_id].add(vec.reshape(1, -1))
#         mem_id = str(uuid.uuid4())[:8]
#         _metadata[user_id].append({"id": mem_id, "content": content})
#         return f"Stored memory {mem_id} for {user_id}."
#     elif action == "retrieve":
#         if not query:
#             return "ERROR: query is required for retrieve."
#         if user_id not in _indexes or _indexes[user_id].ntotal == 0:
#             return f"No memories found for {user_id}."
#         vec = _embed(query, region)
#         k = min(top_k, _indexes[user_id].ntotal)
#         scores, idxs = _indexes[user_id].search(vec.reshape(1, -1), k)
#         results = []
#         for score, idx in zip(scores[0], idxs[0]):
#             if idx >= 0:
#                 results.append(f"[score={score:.3f}] {_metadata[user_id][idx]['content']}")
#         return "\n".join(results) if results else "No relevant memories found."
#     elif action == "list":
#         mems = _metadata.get(user_id, [])
#         if not mems:
#             return f"No memories stored for {user_id}."
#         return "\n".join(f"- [{m['id']}] {m['content']}" for m in mems)
#     else:
#         return f"ERROR: unknown action '{action}'. Use store | retrieve | list."

print("mem0_memory patch confirmed. MEM0_LLM_MODEL =", os.environ["MEM0_LLM_MODEL"])
print("If mem0_memory still fails, uncomment the case_memory backup above.")

In [ ]:
# =============================================================================
# WEEK 15/16 RECAP - RE-DECLARE DATA & TOOLS SO THIS NOTEBOOK STANDS ALONE
# =============================================================================
# Honest note: this data was declared in Week 15 (TRANSACTION_DATABASE and
# CUSTOMER_HISTORY) and Week 16 (the tools). We re-declare here so today's
# notebook runs top-to-bottom without importing anything from prior weeks'
# kernels. In a production codebase this would live in a shared module.

# --- Transaction database (from Week 15, trimmed to the cases we will use) ---
TRANSACTION_DATABASE = {
    "TXN-001": {"id": "TXN-001", "amount": 4500, "merchant": "Unknown Overseas",
                "type": "wire_transfer", "time": "03:47", "location": "Lagos, Nigeria",
                "description": "Wire transfer of $4,500 to unknown overseas account at 3:47 AM."},
    "TXN-002": {"id": "TXN-002", "amount": 89.99, "merchant": "Netflix",
                "type": "subscription", "time": "10:00", "location": "Online",
                "description": "Monthly Netflix subscription."},
    "TXN-005": {"id": "TXN-005", "amount": 2100, "merchant": "Luxury Jewelry Store",
                "type": "in_store", "time": "11:30", "location": "Miami",
                "description": "$2,100 at jewelry store in Miami; customer confirmed in Chicago at time."},
    "TXN-009": {"id": "TXN-009", "amount": 8200, "merchant": "New Payee Transfer",
                "type": "wire_transfer", "time": "02:30", "location": "Foreign IP",
                "description": "Password changed; $8,200 to new payee within 15 minutes; foreign IP."},
    "TXN-042": {"id": "TXN-042", "amount": 12500, "merchant": "International Wire - Russia",
                "type": "wire_transfer", "time": "23:15", "location": "Moscow, Russia",
                "description": "Wire transfer of $12,500 to Moscow-based recipient; no prior international activity."},
}

# --- Customer spending history (from Week 15) ---
CUSTOMER_HISTORY = {
    "CUST-001": {"avg_monthly_spend": 2500, "international_transactions": 0,
                 "account_age_years": 5, "typical_hours": "8:00-22:00"},
    "CUST-002": {"avg_monthly_spend": 3200, "international_transactions": 0,
                 "account_age_years": 3, "typical_hours": "7:00-23:00"},
}

# --- Week 16 specialist tools (Strands @tool decorator) ---

@tool
def lookup_transaction(transaction_id: str) -> str:
    """Look up transaction details by ID from the fraud database.

    Args:
        transaction_id: The transaction ID (e.g., 'TXN-001').
    """
    txn = TRANSACTION_DATABASE.get(transaction_id)
    return json.dumps(txn, indent=2) if txn else f"Transaction {transaction_id} not found."


@tool
def check_customer_history(customer_id: str) -> str:
    """Retrieve customer transaction history and behavioral patterns.

    Args:
        customer_id: The customer ID to look up.
    """
    hist = CUSTOMER_HISTORY.get(customer_id)
    return json.dumps(hist, indent=2) if hist else f"Customer {customer_id} not found."


@tool
def calculate_risk_score(amount: float, is_international: bool,
                         is_unusual_hour: bool, merchant_known: bool) -> str:
    """Calculate a fraud risk score based on transaction characteristics.

    Args:
        amount: Transaction amount in dollars.
        is_international: Whether the transaction crosses borders.
        is_unusual_hour: Whether the transaction occurred at an unusual time.
        merchant_known: Whether the merchant is in the customer's history.
    """
    score = 0
    if amount > 1000:
        score += 30
    if is_international:
        score += 25
    if is_unusual_hour:
        score += 20
    if not merchant_known:
        score += 25
    level = "LOW" if score < 30 else "MEDIUM" if score < 60 else "HIGH"
    return json.dumps({"risk_score": score, "risk_level": level}, indent=2)


print("Week 15/16 recap loaded.")
print(f"  Transactions: {list(TRANSACTION_DATABASE.keys())}")
print(f"  Customers:    {list(CUSTOMER_HISTORY.keys())}")
print(f"  Tools:        lookup_transaction, check_customer_history, calculate_risk_score")

# Section 1: Agentic RAG vs Naive RAG

## Two Shapes of RAG

You've already seen naive RAG. In Week 13 you called `retrieveAndGenerate` and got back an answer. One retrieval, one generation, done.

**Agentic RAG** is different. Retrieval is a tool the agent can call, skip, or call again. The agent decides.

```mermaid
graph LR
    subgraph "Naive RAG (Week 13)"
        Q1[Query] --> R1[Retrieve top-k]
        R1 --> G1[Generate answer]
    end

    subgraph "Agentic RAG (Week 17)"
        Q2[Query] --> A[Agent]
        A -->|decide: need info?| R2[Retrieve top-k]
        R2 --> A
        A -->|decide: need more?| R2
        A --> G2[Generate answer]
    end

    style A fill:#e8f5e9,stroke:#4caf50,stroke-width:2px
```

## Why Not Always Agentic?

Agentic RAG is more powerful, but also more expensive:

| Shape | Cost per query | Latency | Best for |
|-------|---------------|---------|----------|
| **Naive RAG** | 1 LLM call + 1 retrieval | ~1-2 s | Single-hop factual Q&A |
| **Agentic RAG** | 3-10x the tokens (multiple loops) | ~5-15 s | Multi-step reasoning, ambiguous queries, workflows with several retrievers |

Published benchmarks put agentic RAG at **3-10x the token cost** of naive RAG. The quality lift only justifies that spend when the workload actually needs reasoning about what to retrieve. Fraud investigation is exactly that case.

## When Agentic Wins

- "Does this customer's history suggest structuring?" - the agent needs to look up customer history, look up the CTR threshold policy, compare, then answer.
- "What is CTR?" - one retrieval, no reasoning needed. Naive wins.

A smart supervisor routes between the two. Today we build the tools that make that possible.

In [ ]:
# =============================================================================
# DEMO: Naive RAG - One-Shot Retrieval + Generation
# =============================================================================
# This is the Week 13 pattern: boto3 bedrock-agent-runtime client, a single
# retrieve call, format the results into the prompt, ask the LLM to answer.
# No agent, no tools, no loop.

bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)


def naive_rag(question: str, top_k: int = 3) -> str:
    """Classic RAG: retrieve top-k chunks, stuff into prompt, generate."""
    # Step 1: Retrieve
    resp = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": question},
        retrievalConfiguration={
            "vectorSearchConfiguration": {"numberOfResults": top_k}
        },
    )
    # Each result has content.text, location, score, metadata
    chunks = [r["content"]["text"] for r in resp["retrievalResults"]]
    context = "\n\n---\n\n".join(chunks)

    # Step 2: Stuff context into a prompt and generate
    prompt = (
        "You are a fraud policy assistant. Answer using ONLY the context below.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    )
    answer = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 400, "temperature": 0},
    )
    return answer["output"]["message"]["content"][0]["text"]


q = "What are the reporting thresholds for currency transaction reports (CTRs)?"
print("Naive RAG answer:")
print("-" * 70)
print(naive_rag(q))

In [ ]:
# =============================================================================
# DEMO: Agentic RAG - Agent with retrieve as a Tool
# =============================================================================
# Now the same question, but wired through a Strands agent. The agent:
#   1. Reads the question
#   2. Decides whether to call retrieve (it will)
#   3. Looks at results
#   4. Decides if it has enough to answer, or needs another retrieval
#   5. Answers
#
# strands_tools.retrieve reads STRANDS_KNOWLEDGE_BASE_ID and AWS_REGION from
# the environment (we set those in Cell 3).

agentic_rag_agent = Agent(
    model=llm,
    tools=[retrieve],
    system_prompt=(
        "You are a fraud policy assistant. You have access to a Knowledge Base "
        "of BSA/AML, FFIEC, OFAC, and internal fraud procedures via the retrieve tool. "
        "When asked a policy question, call retrieve with a focused search query, "
        "read the results, and answer based ONLY on what the KB returned. "
        "If the KB doesn't cover the question, say so and don't invent an answer."
    ),
)

print("Agentic RAG answer:")
print("-" * 70)
response = agentic_rag_agent(q)
print(response)
print()
print("Notice the agent chose its own query wording for retrieve,")
print("and it can call retrieve again if the first result was poor.")

> **Think About It #1**: Naive RAG costs around one LLM call per query; agentic RAG costs 3 to 10 times that because of the reasoning loop. For a fraud pipeline, which queries deserve agentic retrieval, and which should stay naive? What signals in the incoming question could a router use to decide?

# Section 2: Wiring Bedrock Knowledge Bases into a Strands Agent

## `strands_tools.retrieve` - A First-Class Retrieval Tool

Strands ships a first-class `retrieve` tool that wraps the Bedrock Knowledge Base `Retrieve` API. Under the hood it:

- Creates a `bedrock-agent-runtime` boto3 client for the configured region
- Calls `retrieve` with your query against the configured KB
- Returns chunks with `content.text`, `location` (S3 URI, web URL, or other), `score` (relevance 0.0 to 1.0), and `metadata` (your custom key-value pairs)

```python
from strands import Agent
from strands_tools import retrieve

# The tool reads STRANDS_KNOWLEDGE_BASE_ID and AWS_REGION from the environment.
# You can also set MIN_SCORE_DEFAULT and RETRIEVE_ENABLE_METADATA_DEFAULT.

policy_agent = Agent(model=llm, tools=[retrieve], system_prompt="...")
```

## What the Agent Sees in a Retrieve Result

```mermaid
graph LR
    A[Agent calls retrieve] --> B[Bedrock KB]
    B --> C[retrievalResults array]
    C --> D["content.text<br/>(the chunk)"]
    C --> E["location<br/>(S3 URI etc)"]
    C --> F["score<br/>(0.0 - 1.0)"]
    C --> G["metadata<br/>(custom fields)"]
```

## Why Agents + KBs Beat Agents with Hard-Coded Rules

In Week 16 you had a dict of 7 fraud policies. What happens when:

- Regulations update? You edit Python.
- A new fraud pattern emerges? You edit Python.
- Legal needs an audit trail of "what policy did the agent apply"? You have nothing - the rule is a dict literal.

With a Bedrock KB, the same content lives in S3, and every retrieve result includes `location`. That is your audit trail.

In [ ]:
# =============================================================================
# DEMO: PolicyRetriever Specialist Agent
# =============================================================================
# A specialist agent is just an Agent with a focused system prompt and a
# focused tool set. We give this one ONLY the retrieve tool (+ the class KB).
# It does nothing but answer fraud policy questions against the KB.

policy_retriever_agent = Agent(
    model=llm,
    tools=[retrieve],
    system_prompt=(
        "You are a fraud policy specialist. Your ONLY job is to answer "
        "regulatory and procedural questions by querying the fraud policy "
        "Knowledge Base via the retrieve tool. "
        "Rules: "
        "1) Always call retrieve with a focused query before answering. "
        "2) Answer ONLY from retrieved content - never from your own knowledge. "
        "3) If the KB doesn't cover the question, say 'The policy KB does not "
        "cover this topic' and stop. "
        "4) In your answer, cite the source location(s) returned by retrieve."
    ),
    # Suppress intermediate output so this agent stays clean when wrapped as a tool
    callback_handler=None,
)

# Demo query
policy_q = "What is the structuring red flag - when transactions are split to avoid CTR?"
print("PolicyRetriever answer:")
print("-" * 70)
print(policy_retriever_agent(policy_q))

In [ ]:
# =============================================================================
# DEMO: Inspect the Raw retrieve Output (what the agent actually got)
# =============================================================================
# strands_tools.retrieve wraps its results into a formatted STRING inside a
# Strands ToolResult envelope (status/content/toolUseId). That is great for
# the agent but not great for debugging or auditing.
#
# For raw inspection we call bedrock-agent-runtime.retrieve() directly, which
# returns the native Bedrock 'retrievalResults' list of dicts, each with:
#   content.text  - the chunk
#   location      - S3 URI / web URL / etc
#   score         - 0.0 to 1.0 relevance
#   metadata      - your custom key-value pairs

resp = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID,
    retrievalQuery={"text": "structuring transactions below 10000"},
    retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
)

results = resp.get("retrievalResults", [])
print("Raw retrieval results (top 3):")
print("=" * 70)
for i, item in enumerate(results, 1):
    content = item.get("content", {}).get("text", "")[:200]
    score = item.get("score", "n/a")
    loc = item.get("location", {})
    print(f"\n#{i}  score={score}")
    print(f"    location: {loc}")
    print(f"    text (truncated): {content}...")

print("\n" + "=" * 70)
print("Why a direct boto3 call here? The strands_tools.retrieve tool is")
print("designed for the agent's use - it returns a pre-formatted string.")
print("For debugging or building specialist wrappers (Lab 1), we call the")
print("Bedrock API directly to get the raw chunks with scores and sources.")

## Lab 1: Build a Tunable PolicyRetriever Agent (15 minutes)

### Your Task

Build a specialist PolicyRetriever agent that:
1. Uses a **wrapped** retrieve tool which filters by a minimum relevance score
2. Refuses to answer questions the KB doesn't cover (no hallucination)
3. Returns answers with explicit source citations

### Steps

1. Create a new `@tool` called `retrieve_policy_with_score` that calls the base `retrieve` tool but drops any result below a `min_score` threshold. Pass `enableMetadata=True` so citations are available.
2. Build a `lab1_policy_agent` Agent that uses your wrapped tool (NOT the raw `retrieve`) and a strict system prompt.
3. Test on two queries:
   - In-scope: "When is OFAC screening required for wire transfers?"
   - Out-of-scope: "What was the S&P 500 close today?"

### Expected Output

- In-scope query: a cited answer with source locations.
- Out-of-scope query: the agent politely refuses without inventing content.

### Stretch (for fast finishers)

Modify the wrapped tool to return BOTH the filtered chunks AND a `summary` field that lists unique source document URIs used. Update the agent's prompt to include a "Sources:" footer that lists those URIs verbatim.

### Homework Extension

Re-run the same in-scope query at `min_score = 0.3, 0.5, 0.7`. Record how many chunks each threshold returns and how the answer changes. What is the production tradeoff between recall (low threshold) and precision (high threshold) for a compliance use case where "missed policy" is worse than "extra chunk"?

In [ ]:
# =============================================================================
# LAB 1: TUNABLE POLICYRETRIEVER
# =============================================================================

# Step 1: Wrap retrieve with a min_score filter
@tool
def retrieve_policy_with_score(query: str, min_score: float = 0.4) -> str:
    """Retrieve fraud policy chunks from the KB, filtering by minimum score.

    Args:
        query: The search query.
        min_score: Minimum relevance score (0.0-1.0); chunks below this are dropped.
    """
    filtered = None  # YOUR CODE

    return json.dumps(filtered, indent=2)


# Step 2: Build the specialist agent using your wrapped tool
lab1_policy_agent = None  # YOUR CODE


# Step 3: Test in-scope and out-of-scope
in_scope_answer = None  # YOUR CODE
print("In-scope:")
print(in_scope_answer)

out_of_scope_answer = None  # YOUR CODE
print("\nOut-of-scope:")
print(out_of_scope_answer)

> **Think About It #2**: Your KB was ingested on day 1. It's now day 60 and BSA/AML regulations have been updated. Your agent is confidently citing policies that no longer apply. What mechanisms would you put in place - at ingestion, at retrieval, and in the agent's prompt - to prevent stale confident answers? Which of these are Week 18 concerns (chunking, indexing, evaluation) and which are Week 17 concerns (agent prompt design)?

# Section 3: FAISS-Backed Agent Memory with `strands_tools.mem0_memory`

## Not Every Retrieval Needs a Managed KB

Bedrock KB is the right answer for **shared, versioned, auditable** knowledge - policy docs, regulatory bulletins, internal procedures.

But some things are smaller, more agent-local, and more dynamic:

- "Remember what we decided about TXN-001 yesterday"
- "These are the five patterns flagged by this team this week"
- "Don't forget the user said they hate false positives"

For these, Strands provides `mem0_memory` - backed by **FAISS** for local vector search (the default when no Mem0 Platform key or OpenSearch host is configured). This is the same FAISS library the industry uses for large-scale vector indexes, running locally in your kernel.

```mermaid
graph LR
    A[Agent] -->|store: action| M[mem0_memory tool]
    A -->|retrieve: action| M
    M -->|FAISS backend| F[(FAISS index<br/>on /tmp)]
```

## `mem0_memory` Actions

| Action | What it does |
|--------|-------------|
| `store` | Add text to memory with a `user_id` and/or `agent_id` |
| `retrieve` | Semantic search over stored memories |
| `list` | List memories for a given scope |
| `get` | Fetch a specific memory by ID |
| `delete` | Remove a memory |
| `history` | Change history for a memory |

## Scoping: `user_id` and `agent_id`

Memories are scoped. `user_id="fraud-investigator-alice"` means Alice's memories don't bleed into Bob's. `agent_id="case-history-retriever"` scopes memories to a specific agent role. You can use either or both.

## Honest Note on Storage Location

By default, `mem0_memory` writes FAISS indexes under `/tmp`, which is **ephemeral on SageMaker Studio and Colab**. It survives within your kernel session but vanishes when the instance stops. For production you'd configure a persistent path (covered in the homework extension).

In [ ]:
# =============================================================================
# DEMO: CaseHistoryRetriever - Store & Retrieve Past Case Outcomes
# =============================================================================
# Scenario: over time, fraud investigators build up a mental library of
# "we've seen this pattern before - last month's TXN-088 looked just like
# this." We give the agent the same capability via mem0.

# Silence mem0's internal logger - it emits benign JSON-parse warnings and
# model-availability notices even on successful runs.
import logging
logging.getLogger("mem0").setLevel(logging.CRITICAL)

# Create a CaseHistoryRetriever specialist agent
case_history_agent = Agent(
    model=llm,
    tools=[mem0_memory],
    system_prompt=(
        "You are a case history specialist. "
        "You store and retrieve past fraud case outcomes using mem0_memory. "
        "When asked about past cases, use action='retrieve' to search. "
        "When told to remember a case, use action='store'. "
        "Always scope to user_id='fraud-team-v1'. "
        "When retrieving, summarize patterns across results - do not just "
        "return raw matches."
    ),
    callback_handler=None,
)

# Seed memory with a few past-case outcomes (in a real system these would be
# pushed by the supervisor after each investigation concludes)
seed_cases = [
    "Case TXN-088: $4,800 wire transfer to Lagos Nigeria at 3AM. BLOCKED. "
    "Customer confirmed fraud. OFAC screening would have flagged on ingest.",
    "Case TXN-091: $2,100 jewelry purchase in Miami while customer confirmed "
    "in Chicago. BLOCKED. Customer had no travel alert set.",
    "Case TXN-093: $89 Netflix subscription. APPROVED. 18-month subscription "
    "history, consistent amount, regular timing.",
    "Case TXN-099: Password change followed by $8,500 transfer to new payee "
    "within 10 minutes from foreign IP. BLOCKED. Classic account takeover.",
]

for case in seed_cases:
    case_history_agent(f"Remember this case for future reference: {case}")

print("Seeded case history memory with 4 past cases.")
print()

# Now test retrieval
recall = case_history_agent(
    "Have we seen similar cases involving wire transfers to overseas "
    "accounts at unusual hours? What were the outcomes?"
)
print("CaseHistoryRetriever answer:")
print("-" * 70)
print(recall)

## Lab 2: Per-Investigator Case History (15 minutes)

### Your Task

Extend the CaseHistoryRetriever so different investigators have isolated memory. Alice shouldn't see Bob's notes, and vice versa.

### Steps

1. Create a specialist agent `scoped_case_agent` that takes the investigator identity as part of each conversation. The agent prompt instructs it to always scope mem0 operations to a specific `user_id`.
2. Store a case for `user_id="alice"` ("TXN-201 blocked - velocity pattern").
3. Store a different case for `user_id="bob"` ("TXN-202 approved - known merchant, regular timing").
4. Query as Alice: should only see her case.
5. Query as Bob: should only see his case.

### Expected Output

- Alice's retrieval returns only TXN-201
- Bob's retrieval returns only TXN-202
- No cross-contamination

### Stretch (for fast finishers)

After both stores, have the agent retrieve from a combined scope (e.g. `user_id="team-shared"`) and show how a team-shared store differs from per-investigator stores. When would you pick per-user vs. team-shared mem0 for a fraud team?

### Homework Extension

Configure `mem0_memory` to persist to a directory OUTSIDE `/tmp` (pick a path inside your SageMaker Studio home). Store a case, restart the kernel, verify the case survives. What production tradeoffs does persistent local memory introduce (backup, cross-instance sharing, GDPR "right to be forgotten")?

In [ ]:
# =============================================================================
# LAB 2: PER-INVESTIGATOR CASE HISTORY
# =============================================================================

# Step 1: Build a case history agent that scopes by investigator user_id.
# The agent should accept the user_id as part of its instruction context.
scoped_case_agent = None  # YOUR CODE


# Step 2: Store Alice's case
alice_store_response = None  # YOUR CODE


# Step 3: Store Bob's case
bob_store_response = None  # YOUR CODE


# Step 4: Query as Alice - should only see TXN-201
alice_recall = None  # YOUR CODE
print("Alice's recall:")
print(alice_recall)


# Step 5: Query as Bob - should only see TXN-202
bob_recall = None  # YOUR CODE
print("\nBob's recall:")
print(bob_recall)

> **Think About It #3**: mem0 writes FAISS to `/tmp` by default, and the SageMaker Studio instance erases `/tmp` when it stops. List three production failure modes this causes, and three mitigations. Which mitigation belongs to the agent layer (what we're doing this week), and which belongs to the MLOps layer (Weeks 19-20)?

# Section 4: Multi-Retriever Fraud Supervisor (Main Outcome)

## The Goal for Today

Extend the Week 16 fraud supervisor so that:

1. The hard-coded `FRAUD_POLICIES` dict and `check_fraud_policy` tool are REPLACED by a `PolicyRetriever` specialist backed by the Bedrock KB.
2. A NEW `CaseHistoryRetriever` specialist is added, backed by FAISS mem0.
3. The supervisor learns to route: policy questions to PolicyRetriever, case-history questions to CaseHistoryRetriever, transactions to Triage / Investigation / Decision as before.

## Supervisor Architecture

```mermaid
graph TD
    USER[User Query]
    SUP[Supervisor<br/>max_iterations=5<br/>max_execution_time=30]

    RT[run_triage]
    RI[run_investigation]
    RP[run_policy_retriever<br/>NEW]
    RC[run_case_history_retriever<br/>NEW]
    RD[run_decision]

    USER --> SUP
    SUP --> RT
    SUP --> RI
    SUP --> RP
    SUP --> RC
    SUP --> RD

    RP -->|retrieve tool| KB[(Bedrock KB)]
    RC -->|mem0_memory tool| F[(FAISS mem0)]

    style RP fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
    style RC fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
    style SUP fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
```

## Iteration Budget - Why `max_iterations` Matters

Agentic RAG can loop forever if you let it. The Strands production recipe is:

```python
supervisor = Agent(
    model=llm,
    tools=[...],
    system_prompt="...",
    max_iterations=5,        # hard cap on tool-call loops
    max_execution_time=30,   # hard wall-clock ceiling
)
```

Without these, a confused supervisor can chain 20 retrievals and burn through your Bedrock budget. The `max_iterations=5` pattern is called "non-negotiable for production" in the Strands docs, and most convergence in agentic RAG happens in iterations 1-3 anyway.

In [ ]:
# =============================================================================
# DEMO: Wrap the Two Retriever Agents as Callable Tools
# =============================================================================
# Same pattern as Week 16's run_triage / run_investigation / run_decision.
# Each @tool is a thin wrapper that forwards to its specialist agent.

@tool
def run_policy_retriever(question: str) -> str:
    """Ask the fraud policy specialist a question against the Bedrock KB.

    Args:
        question: Natural-language question about fraud policies,
                  BSA/AML, OFAC, CTR rules, or internal procedures.
    """
    return str(policy_retriever_agent(question))


@tool
def run_case_history_retriever(question: str,
                               investigator_id: str = "fraud-team-v1") -> str:
    """Ask the case history specialist about past fraud cases.

    Args:
        question: Natural-language question about past cases or patterns.
        investigator_id: Which investigator's memory scope to use.
    """
    return str(case_history_agent(
        f"INVESTIGATOR_ID={investigator_id}. {question}"
    ))


# Also re-declare the Week 16 triage / investigation / decision specialists
# at the same level of abstraction. We keep these minimal for the demo.

triage_agent = Agent(
    model=llm,
    tools=[lookup_transaction, calculate_risk_score],
    system_prompt=(
        "You classify transactions by risk level (LOW/MEDIUM/HIGH). "
        "Look up the transaction, calculate risk, return a JSON verdict."
    ),
    callback_handler=None,
)

investigation_agent = Agent(
    model=llm,
    tools=[lookup_transaction, check_customer_history],
    system_prompt=(
        "You gather evidence for flagged transactions. Look up transaction "
        "and customer history and produce an evidence report."
    ),
    callback_handler=None,
)

decision_agent = Agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You render a final verdict: APPROVE, FLAG_FOR_REVIEW, or BLOCK. "
        "Use the triage, investigation, and policy/history context you're given."
    ),
    callback_handler=None,
)


@tool
def run_triage(transaction_id: str) -> str:
    """Classify a transaction's risk level.

    Args:
        transaction_id: The transaction to triage.
    """
    return str(triage_agent(f"Triage transaction {transaction_id}."))


@tool
def run_investigation(transaction_id: str, customer_id: str) -> str:
    """Investigate a flagged transaction.

    Args:
        transaction_id: The transaction to investigate.
        customer_id:    The customer who owns it.
    """
    return str(investigation_agent(
        f"Investigate transaction {transaction_id} for customer {customer_id}."
    ))


@tool
def run_decision(triage_result: str, investigation_result: str,
                 policy_context: str, history_context: str) -> str:
    """Render the final verdict.

    Args:
        triage_result:        Triage classification JSON.
        investigation_result: Investigation evidence report.
        policy_context:       Policy guidance from the PolicyRetriever.
        history_context:      Past-case context from the CaseHistoryRetriever.
    """
    return str(decision_agent(
        f"Triage: {triage_result}\n\n"
        f"Investigation: {investigation_result}\n\n"
        f"Policy context: {policy_context}\n\n"
        f"Case history: {history_context}\n\n"
        "Render the final verdict."
    ))

print("All 5 specialist wrappers declared:")
print("  run_triage, run_investigation, run_policy_retriever,")
print("  run_case_history_retriever, run_decision")

In [ ]:
# =============================================================================
# DEMO: Supervisor Routing
# =============================================================================
# We build the supervisor with all 5 specialist tools (triage, investigation,
# policy_retriever, case_history_retriever, decision) and iteration caps.
# Then we ask two questions to show routing:
#   Q1: policy-only   -> should route to run_policy_retriever
#   Q2: full workflow -> should route through several specialists

supervisor = Agent(
    model=llm,
    tools=[
        run_triage,
        run_investigation,
        run_policy_retriever,
        run_case_history_retriever,
        run_decision,
    ],
    system_prompt=(
        "You are the fraud operations supervisor at a major financial institution. "
        "Route each query to the appropriate specialist(s): "
        "  - Pure policy/regulation questions -> run_policy_retriever only "
        "  - Past-case questions -> run_case_history_retriever only "
        "  - Transaction investigations -> run_triage, then run_investigation "
        "    (if risk is MEDIUM/HIGH), then run_policy_retriever AND "
        "    run_case_history_retriever for context, then run_decision "
        "Do not call specialists you don't need. Conserve tool calls."
    ),
    max_iterations=5,        # hard cap on reasoning loops
    max_execution_time=30,   # 30-second wall clock
)

# Query 1: policy-only - supervisor should call ONLY run_policy_retriever
print("=" * 70)
print("QUERY 1 (policy-only): 'What triggers a CTR filing requirement?'")
print("=" * 70)
print(supervisor("What triggers a CTR filing requirement?"))
print()

# Query 2: full workflow - supervisor should orchestrate multiple specialists
print("=" * 70)
print("QUERY 2 (full investigation): 'Investigate TXN-042 for CUST-001.'")
print("=" * 70)
print(supervisor(
    "Investigate transaction TXN-042 for customer CUST-001. "
    "Render a final verdict with policy justification and case-history context."
))

## Lab 3 (MAIN OUTCOME): Retire `FRAUD_POLICIES` - Multi-Retriever Supervisor (15 minutes)

### Your Task

This is the Week 17 main outcome. Build a production-ready fraud supervisor that replaces Week 16's hard-coded `FRAUD_POLICIES` dict with the `PolicyRetriever` you built in Section 2, adds the `CaseHistoryRetriever` from Section 3, and routes correctly across both.

### Steps

1. Build `lab3_supervisor` with exactly these tools: `run_triage`, `run_investigation`, `run_policy_retriever`, `run_case_history_retriever`, `run_decision`
2. Do NOT include Week 16's old `check_fraud_policy` tool - it's deprecated.
3. Set `max_iterations=5` and `max_execution_time=30`.
4. Write a system prompt that makes routing explicit:
   - Policy/regulation questions -> policy retriever only
   - Case-history questions -> case-history retriever only
   - Transaction investigations -> full pipeline with both retrievers contributing context to `run_decision`
5. Test on three queries covering all three routing paths.

### Test Queries

1. "What are the OFAC sanctions screening requirements?" (policy-only)
2. "Have we seen wire transfers to Russia before?" (case-history-only)
3. "Investigate TXN-042 for CUST-001. Render a full verdict." (full pipeline)

### Expected Output

- Each query routes correctly
- Query 3's final decision cites BOTH policy guidance AND past-case context
- No `check_fraud_policy` tool appears anywhere

### Stretch (for fast finishers)

Add a `log_routing` `@tool` that records which specialists the supervisor called for each query. Require the supervisor to call `log_routing` LAST for every query, listing the specialist names. Use it to audit the three test queries above - does the supervisor actually route the way you told it to?

### Homework Extension

1. Add a fourth specialist: a `guardrail_check` agent backed by a second mem0 store for OFAC sanctions lists specifically. The supervisor must call it on any transaction involving international transfers.
2. Measure token usage and latency for 5 queries of varying complexity. Plot cost-per-query vs. query-complexity. Identify the crossover point where naive RAG would have been cheaper.
3. (Connects to Week 14): wrap your fine-tuned DistilBERT fraud classifier from Week 14 as a `distilbert_classifier` specialist tool and include it as a fast pre-filter. What cost savings does this produce?

In [ ]:
# =============================================================================
# LAB 3 (MAIN OUTCOME): MULTI-RETRIEVER FRAUD SUPERVISOR
# =============================================================================

# Step 1: Build the production-grade supervisor with iteration budgets
lab3_supervisor = None  # YOUR CODE


# Step 2: Test query 1 - policy-only
q1_answer = None  # YOUR CODE
print("=" * 70)
print("QUERY 1: policy-only")
print("=" * 70)
print(q1_answer)


# Step 3: Test query 2 - case-history-only
q2_answer = None  # YOUR CODE
print("\n" + "=" * 70)
print("QUERY 2: case-history-only")
print("=" * 70)
print(q2_answer)


# Step 4: Test query 3 - full pipeline
q3_answer = None  # YOUR CODE
print("\n" + "=" * 70)
print("QUERY 3: full pipeline")
print("=" * 70)
print(q3_answer)

> **Think About It #4**: Your supervisor routes based on the question text. For Query 3 ("Investigate TXN-042..."), what happens if the LLM mis-reads the intent and only calls `run_policy_retriever`? The user gets a policy summary with no verdict.
>
> In production, you'd add self-correction: the supervisor detects its own insufficient answer and re-runs with better routing. That's called **query rewriting / corrective RAG**, and it's one of the Week 18 topics. For now: what's the cheapest guardrail you could add to Week 17's supervisor to catch obviously wrong routing BEFORE returning to the user?

# Summary: What We Learned Today

## Key Takeaways

### Agentic RAG
- Retrieval is a tool, not a pipeline step. The agent decides when and whether.
- Cost: 3-10x naive RAG per query. Budget with `max_iterations` and `max_execution_time`.

### `strands_tools.retrieve` with Bedrock KB
- Reads `STRANDS_KNOWLEDGE_BASE_ID` and `AWS_REGION` from env.
- Returns `content.text`, `location`, `score`, `metadata` per chunk.
- Perfect for shared, versioned, auditable corpora (regulations, policies).

### `strands_tools.mem0_memory` with FAISS
- Local, persistent-within-kernel agent memory with `user_id` and `agent_id` scoping.
- Great for per-agent or per-investigator notes.
- Honest caveat: default `/tmp` storage is ephemeral on SageMaker Studio.

### Multi-Retriever Supervisor
- Same agents-as-tools pattern as Week 16, now with retrieval specialists.
- The hard-coded `FRAUD_POLICIES` dict from Week 16 has been retired.
- Iteration budgets (`max_iterations=5`, `max_execution_time=30`) are the production-required control for agentic RAG cost.

## Looking Ahead

- **Week 18** takes the retrieval pipeline itself apart: chunking strategies (fixed, semantic, recursive), reranking (cross-encoders, Cohere Rerank), and RAG evaluation with RAGAS (faithfulness, answer relevancy, context precision/recall). Everything we built today, we'll measure and improve.
- **Weeks 19-20** add MLOps to these exact agents - DVC for data versioning, MLflow for experiment tracking, Langfuse/LiteLLM for LLM-specific observability (tracking cost, latency, and quality of every supervisor call in production).
- **Weeks 21-22** orchestrate re-ingestion of KB documents on a schedule using Airflow/MWAA so your policy KB stays fresh.
- **Week 23** connects today's citations (from `retrieve`'s `location` field) to explainability and audit requirements under GDPR / EU AI Act.
- **Week 24 capstone**: a retrieval-augmented multi-agent system is one of the three canonical capstone paths. Everything you built today is a starting point.

## Homework

### Homework 1: Score Threshold Tuning (Lab 1 extension)
Re-run Lab 1's in-scope query at `min_score = 0.3, 0.5, 0.7`. Record the number of chunks returned and the answer quality. For a compliance use case where "missed policy" is worse than "extra chunk", which threshold is safest? Why?

### Homework 2: Persistent mem0 (Lab 2 extension)
Configure `mem0_memory` to persist to a directory inside your SageMaker Studio home (not `/tmp`). Store a case, restart the kernel, verify the case survives. Document the GDPR / compliance implications of persistent local agent memory.

### Homework 3: Fine-Tuned Model as Specialist (connects to Week 14)
Load your Week 14 fine-tuned DistilBERT fraud classifier. Wrap it as a Strands `@tool` that takes a transaction description and returns `{label, confidence}`. Add it to the Lab 3 supervisor as a fast pre-filter: if DistilBERT is confident, skip the full pipeline.

### Homework 4: Cost Instrumentation (Lab 3 extension)
Measure tokens in / tokens out and wall-clock latency for 5 queries of varying complexity. Plot cost-per-query vs. query-complexity. Identify the break-even point where naive RAG would have been cheaper than agentic RAG.

## Great Work Today

**The journey so far:**
- Weeks 11-14: Make LLMs respond (prompt, fine-tune, classify)
- Weeks 15-16: Make LLMs ACT (tools, memory, supervisor multi-agent)
- **Week 17: Make LLMs LOOK THINGS UP - agentic RAG with Bedrock KB + FAISS**

**Next up**: Week 18 - what's inside that retrieval pipeline.